In [60]:
from pyspark.sql import SparkSession

from pyspark.sql import SparkSession
from pyspark.ml.feature import Word2Vec
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, ArrayType, StringType
from pyspark.sql.functions import explode, col, count, desc, rank
import pandas as pd

In [25]:
spark = (SparkSession
         .builder
         .master('local[*]')
         .getOrCreate())

# Helper Functions

In [30]:
def compute_k_silhouette(filepath, k_range=range(5, 100, 5), vector_size=100):
    """
    Given a parquet file, compute silhouette scores for a range of KMeans cluster sizes.

    Returns a dictionary {k: silhouette_score}.
    """
    spark = SparkSession.builder.appName("Amenities_Clustering").getOrCreate()
    
    # Define schema
    schema = StructType([
        StructField("id", IntegerType(), True),
        StructField("price", DoubleType(), True),
        StructField("amenities_array", ArrayType(StringType()), True)
    ])
    
    # Load data
    df = spark.read.schema(schema).parquet(filepath)
    df = df.dropna(subset=['amenities_array'])
    
    # Word2Vec
    word2vec = Word2Vec(inputCol="amenities_array", outputCol="amenities_vec", vectorSize=vector_size, minCount=1)
    wv_model = word2vec.fit(df)
    df_vec = wv_model.transform(df)
    
    # Silhouette evaluator
    evaluator = ClusteringEvaluator(featuresCol="amenities_vec", metricName="silhouette", distanceMeasure="squaredEuclidean")
    
    # Compute silhouette scores for each k
    k_silhouette = {}
    for k in k_range:
        kmeans = KMeans(featuresCol="amenities_vec", k=k, seed=42)
        model = kmeans.fit(df_vec)
        predictions = model.transform(df_vec)
        silhouette = evaluator.evaluate(predictions)
        print(f"k={k}, Silhouette={silhouette}")
        k_silhouette[k] = silhouette
    
    return k_silhouette


In [45]:
def get_top_amenities(filepath, k=45, vector_size=100, top_n=10):
    """
    Given a parquet file and a KMeans k, return the top N amenities per cluster as a Pandas DataFrame.
    """
    spark = SparkSession.builder.appName("Amenities_Clustering").getOrCreate()
    
    # Define schema
    schema = StructType([
        StructField("id", IntegerType(), True),
        StructField("price", DoubleType(), True),
        StructField("amenities_array", ArrayType(StringType()), True)
    ])
    
    # Load data
    df = spark.read.schema(schema).parquet(filepath)
    df = df.dropna(subset=['amenities_array'])
    
    # Word2Vec
    word2vec = Word2Vec(inputCol="amenities_array", outputCol="amenities_vec", vectorSize=vector_size, minCount=1)
    wv_model = word2vec.fit(df)
    df_vec = wv_model.transform(df)
    
    # KMeans clustering
    kmeans = KMeans(featuresCol="amenities_vec", k=k, seed=42)
    model = kmeans.fit(df_vec)
    predictions = model.transform(df_vec)
    
    # Explode amenities
    exploded = predictions.select("prediction", explode(col("amenities_array")).alias("amenity"))
    
    # Count frequency per cluster
    amenity_counts = exploded.groupBy("prediction", "amenity").count()
    
    # Rank top amenities per cluster
    window = Window.partitionBy("prediction").orderBy(desc("count"))
    top_amenities = amenity_counts.withColumn("rank", row_number().over(window)) \
                                  .filter(col("rank") <= top_n) \
                                  .orderBy("prediction", "rank")
    
    # Convert to Pandas for easier viewing
    return top_amenities.toPandas()


In [57]:
def top_clusters(filepath, k=45, vector_size=100, top_n=10):
    """
    Given a parquet file and a KMeans k, return the top N amenities per cluster
    as a Pandas DataFrame, ordered by cluster size.
    """
    spark = SparkSession.builder.appName("Amenities_Clustering").getOrCreate()
    
    # --- Data Loading and Preparation ---
    schema = StructType([
        StructField("id", IntegerType(), True),
        StructField("price", DoubleType(), True),
        StructField("amenities_array", ArrayType(StringType()), True)
    ])
    
    df = spark.read.schema(schema).parquet(filepath)
    df = df.dropna(subset=['amenities_array'])
    
    # --- Word2Vec (Skipped for amenity-based clustering, but kept in code flow) ---
    # *Note: For word frequency, the vector calculation is not strictly needed, 
    # but we keep it to maintain the clustering on the 'amenities_vec' feature.*
    word2vec = Word2Vec(inputCol="amenities_array", outputCol="amenities_vec", vectorSize=vector_size, minCount=1)
    wv_model = word2vec.fit(df)
    df_vec = wv_model.transform(df)
    
    # --- KMeans clustering ---
    kmeans = KMeans(featuresCol="amenities_vec", k=k, seed=42)
    model = kmeans.fit(df_vec)
    predictions = model.transform(df_vec) # predictions now has 'prediction' (cluster ID) and 'amenities_array'
    
    # --- Analysis: Get Top Amenities per Cluster ---
    
    # 1. Explode the amenities array so each row has one cluster ID and one amenity
    df_exploded = predictions.select("prediction", explode(col("amenities_array")).alias("amenity"))
    
    # 2. Calculate the count of each (cluster, amenity) pair
    amenity_counts = df_exploded.groupBy("prediction", "amenity").agg(count("*").alias("amenity_count"))
    
    # 3. Calculate the size of each cluster
    cluster_sizes = amenity_counts.groupBy("prediction").agg(
        count(col("amenity")).alias("cluster_size"), # This counts all unique amenity entries in the cluster
        count(col("prediction")).alias("total_amenity_occurrences_in_cluster") # More accurate cluster size representation
    )
    
    # 4. Join counts with cluster sizes and sort by cluster size (descending)
    df_results = amenity_counts.join(cluster_sizes.select("prediction", "total_amenity_occurrences_in_cluster"), on="prediction") \
        .orderBy(desc("total_amenity_occurrences_in_cluster"), col("prediction"), desc("amenity_count"))
    
    # 5. Use a Window function to rank amenities within each cluster
    from pyspark.sql.window import Window
    window_spec = Window.partitionBy("prediction").orderBy(desc("amenity_count"))
    
    df_ranked = df_results.withColumn("rank", rank().over(window_spec))
    
    # 6. Filter for the top N amenities and convert to Pandas for final output
    final_df = df_ranked.filter(col("rank") <= top_n) \
            .orderBy(desc("total_amenity_occurrences_in_cluster"), col("rank")) \
            .select(
                col("prediction").alias("Cluster_ID"),
                col("total_amenity_occurrences_in_cluster").alias("Cluster_Size"),
                col("amenity").alias("Top_Amenity"),
                col("amenity_count").alias("Amenity_Frequency"),
                col("rank").alias("Rank_In_Cluster")
            ) \
            .toPandas()

    return final_df

# Example of how to use the function:
# result_df = top_clusters("path/to/your/file.parquet", k=50, top_n=5)
# print(result_df)

# New York

In [31]:
ny_filepath = 'data/cleaned_array/new-york-city.parquet'
k_silhouette_dict = compute_k_silhouette(ny_filepath)

k=5, Silhouette=0.10675944637364297
k=10, Silhouette=0.09668455858212036
k=15, Silhouette=0.29416445425279797
k=20, Silhouette=0.4992682413972435
k=25, Silhouette=0.5451819757991989
k=30, Silhouette=0.5854062325011551
k=35, Silhouette=0.7020301839101424
k=40, Silhouette=0.7361421648718259
k=45, Silhouette=0.7910561118369603
k=50, Silhouette=0.7970501869138599
k=55, Silhouette=0.8076981791521826
k=60, Silhouette=0.8076882403716916
k=65, Silhouette=0.8209910684621484
k=70, Silhouette=0.8440580728593599
k=75, Silhouette=0.8499462022847001
k=80, Silhouette=0.8617403431919773
k=85, Silhouette=0.8768313030561621
k=90, Silhouette=0.8796764806654562
k=95, Silhouette=0.8727938300467124


In [54]:
top_clusters(ny_filepath, k=45, vector_size=100, top_n=10)

,Cluster_ID,Cluster_Size,Top_Amenity,Amenity_Frequency,Rank_In_Cluster
0,44,1013,Stove,2009,1
1,44,1013,Bed linens,1687,2
2,44,1013,Clothing storage,1634,3
3,44,1013,Laundromat nearby,1501,4
4,44,1013,Mosquito net,1324,5
...,...,...,...,...,...
123,37,1,Hot tub,751,1
124,40,1,Private room,1105,1
125,41,1,Portable fans,3095,1
126,42,1,Washer,718,1


# Los Angeles

In [39]:
la_filepath = 'data/cleaned_array/los-angeles.parquet'
k_silhouette_dict_la = compute_k_silhouette(la_filepath)

k=5, Silhouette=-0.01001398489786795
k=10, Silhouette=0.1669820160169833
k=15, Silhouette=0.3513109949094823
k=20, Silhouette=0.46411967677300686
k=25, Silhouette=0.5248507997258086
k=30, Silhouette=0.5675258815006594
k=35, Silhouette=0.685991038660461
k=40, Silhouette=0.6984335991132571
k=45, Silhouette=0.7188380357600834
k=50, Silhouette=0.7895169630467851
k=55, Silhouette=0.8060561090108
k=60, Silhouette=0.8191952725629985
k=65, Silhouette=0.8289343152864048
k=70, Silhouette=0.8409631240427172
k=75, Silhouette=0.8408621235929822
k=80, Silhouette=0.8407904168258067
k=85, Silhouette=0.8407972756243709
k=90, Silhouette=0.8659739671880691
k=95, Silhouette=0.866072527672203


In [55]:
top_clusters(la_filepath, k=50, vector_size=100, top_n=10)

,Cluster_ID,Cluster_Size,Top_Amenity,Amenity_Frequency,Rank_In_Cluster
0,30,1128,Breakfast,1927,1
1,30,1128,Patio or balcony,1801,2
2,30,1128,Hot water,1673,3
3,30,1128,BBQ grill,1568,4
4,30,1128,Dining table,1524,5
...,...,...,...,...,...
128,41,1,Stove,6916,1
129,44,1,Self check-in,5541,1
130,45,1,Washer,6133,1
131,48,1,Barbecue utensils,3208,1


# Chicago

In [41]:
chi_filepath = 'data/cleaned_array/chicago.parquet'
k_silhouette_dict_chi = compute_k_silhouette(chi_filepath)

k=5, Silhouette=0.23915742544115468
k=10, Silhouette=0.365201170530704
k=15, Silhouette=0.48133645031837374
k=20, Silhouette=0.5909645799776148
k=25, Silhouette=0.6595543419445551
k=30, Silhouette=0.7004957587217137
k=35, Silhouette=0.714022477964166
k=40, Silhouette=0.7636928491752046
k=45, Silhouette=0.801830145697007
k=50, Silhouette=0.828641839887412
k=55, Silhouette=0.8475832655651351
k=60, Silhouette=0.8495937082879641
k=65, Silhouette=0.8755831826795972
k=70, Silhouette=0.8827340912437022
k=75, Silhouette=0.901323342205457
k=80, Silhouette=0.9074195608028769
k=85, Silhouette=0.9116658797424746
k=90, Silhouette=0.9148414445447355
k=95, Silhouette=0.9178591873669925


In [56]:
top_clusters(la_filepath, k=45, vector_size=100, top_n=10)

,Cluster_ID,Cluster_Size,Top_Amenity,Amenity_Frequency,Rank_In_Cluster
0,27,1220,Hair dryer,5695,1
1,27,1220,Free dryer - In unit,3774,2
2,27,1220,Keypad,3198,3
3,27,1220,Cable TV,2884,4
4,27,1220,Breakfast,1927,5
...,...,...,...,...,...
122,32,1,Barbecue utensils,3208,1
123,33,1,Heating,13950,1
124,34,1,Dishes and silverware,7047,1
125,35,1,Refrigerator,501,1
